# VC1 · Demo en directo — Smart City
## "¿Tenemos un problema de NO₂ en el Ensanche?"

**Esto NO es la actividad evaluable.** Es la demostración de la videoconferencia del NF1.
Tu actividad va de una plataforma de *gaming* y usa otros datos, otras columnas y otros
umbrales. Aquí puedes ejecutar, romper y experimentar libremente: **no se entrega**.

**Antes de ejecutar nada**, una sola vez, desde la raíz del repositorio:

```bash
python demo/nf1_smartcity/preparar_demo.py
```

---

### El encargo

El área de Movilidad ha preguntado esta mañana:

> *"Los vecinos del Ensanche dicen que se ahogan. ¿Es verdad? ¿Tenemos un problema de NO₂ ahí?
> Tenemos sensores desde hace dos años, ¿no? Dime algo."*

Tenemos los datos. Vamos a responder. Ahora.

In [ ]:
import json, os, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd

# Anclamos las rutas a la carpeta de ESTE cuaderno, no al directorio de trabajo.
# Asi el cuaderno encuentra sus datos tanto si Jupyter lo ejecuta desde su propia
# carpeta (lo habitual) como si lo lanzas desde la raiz del repo.
try:
    BASE = Path(__file__).parent                 # por si se ejecuta como script
except NameError:
    BASE = Path.cwd()
    if BASE.name != "nf1_smartcity":             # cwd = raiz del repo -> bajamos
        BASE = BASE / "demo" / "nf1_smartcity"

RAW, BENCH, PROC = BASE / "raw", BASE / "bench", BASE / "proc"

mb = lambda p: round(os.path.getsize(p) / 1024**2, 1)

def cronometra(f):
    t = time.perf_counter()
    r = f()
    return round(time.perf_counter() - t, 2), r

assert (RAW / "sensores.csv").exists(), (
    "No encuentro los datos. Ejecuta una vez, desde la raiz del repo:\n"
    "    python demo/nf1_smartcity/preparar_demo.py"
)
print("Datos listos en:", BASE)


---
# ACTO 1 · Respondemos con lo que tenemos

Los datos están. Están en `raw/`, tal y como los escupe el sensor.
Cargamos y contestamos a Movilidad. No debería llevar ni dos minutos.

In [ ]:
df = pd.read_csv(f"{RAW}/sensores.csv")
print(df.shape)
df.head()

### La respuesta a Movilidad, en una línea

In [ ]:
# "NO2 medio por estación". Es literalmente la pregunta que nos han hecho.
df.groupby("estacion")["no2"].mean().round(1).sort_values(ascending=False)

**Y aquí se acaba la demo, ¿no?**

No. Mira bien esa tabla antes de seguir. Hay **dos cosas que la invalidan por completo**, y las
dos están a la vista:

1. `Ensanche` y `ensanche` aparecen como **dos estaciones distintas**... con medias distintas.
   Y `Centro`, `CENTRO`, `centro`, ` Centro` son cuatro. Tenemos **4 estaciones reales
   escritas de 11 formas**, así que estamos promediando trozos aleatorios de la misma estación.
2. Los valores son **absurdos**. No hay ninguna ciudad del planeta con un NO₂ medio de 280.

Si mandas esa tabla a Movilidad, te la van a creer. Ese es el problema.

### ¿De dónde salen esos números imposibles?

In [ ]:
df.info()   # mira el tipo de 'ruido_db'... ¿por qué es texto?

In [ ]:
print("valores raros en ruido_db:", [v for v in df["ruido_db"].unique() if not v.replace(".", "").isdigit()][:5])
print()
print("nulos en no2      :", int(df["no2"].isna().sum()))
print("máximo de no2     :", df["no2"].max(), "  <-- un sensor averiado reportando 9999")
print("estaciones únicas :", df["estacion"].nunique(), " (reales: 4)")

Tres patologías distintas en cinco columnas:

- **Un centinela de texto** (`ERROR`) dentro de `ruido_db`: una sola celda de texto convierte
  **la columna entera** en texto. No puedes ni sumarla.
- **Huecos** (`NaN`) en `no2`: el sensor no reportó.
- **Valores imposibles** (`9999`): el sensor reportó, pero estaba averiado. Es el peor de los
  tres, porque **parece un dato**.

### El número que hay que entender (esto ha caído en examen)

In [ ]:
s = pd.to_numeric(df["no2"], errors="coerce")

print("media   (columna entera):", round(s.mean(), 1))
print("mediana (columna entera):", round(s.median(), 1))
print("mediana (solo 0-400)    :", round(s[s.between(0, 400)].median(), 1))
print()
print("valores imposibles      :", int((s > 400).sum()), "de", len(s), "->", round(100*(s>400).sum()/len(s), 1), "%")

**Un 1,2 % de sensores averiados ha cuadruplicado la media.** La mediana ni se inmuta.

Ahí tienes, en tres números, la respuesta a una pregunta clásica de examen: *¿por qué imputar
con la mediana y no con la media?* Porque la media es una **suma**: un solo `9999` la arrastra.
La mediana es una **posición**: le da igual cuánto vale el valor extremo, solo cuántos hay.

Y ahora el segundo número, que es el que se escapa: fíjate en que **la mediana en rango y la
mediana entera casi coinciden**. Podrías pensar que da igual filtrar. **No da igual**: aunque
imputes bien los huecos, esos `9999` **siguen dentro de la tabla**. Habrías rellenado los
nulos con una mediana impecable sobre un dataset que sigue mintiendo.

> **El orden importa:** primero decides qué es imposible → lo marcas como ausente → y solo
> entonces calculas con lo que queda. Saltarte un paso no da error. Da un número creíble y
> falso, que es mucho peor.

**No podemos responder a Movilidad.** Necesitamos el histórico limpio. Y ahí aparece el
siguiente problema: son dos años de datos.

---
# ACTO 2 · Ahora a escala: ¿dónde guardamos esto?

Supongamos el trabajo de limpieza hecho *(eso es exactamente lo que harás tú en la actividad,
con los datos del juego)*. El histórico limpio son **1.000.000 de lecturas**.

Están guardadas en `bench/`, en **los dos formatos**, para poder compararlos. Y no te voy a
contar cuál es mejor: **lo vamos a medir**, aquí, en este Codespace de 2 núcleos —el mismo que
vas a abrir tú—.

In [ ]:
print("CSV     :", mb(f"{BENCH}/lecturas.csv"), "MB")
print("Parquet :", mb(f"{BENCH}/lecturas.parquet"), "MB")
print("ratio   :", round(mb(f"{BENCH}/lecturas.csv") / mb(f"{BENCH}/lecturas.parquet"), 1), "x")

In [ ]:
t_csv, d_csv = cronometra(lambda: pd.read_csv(f"{BENCH}/lecturas.csv"))
t_pq,  d_pq  = cronometra(lambda: pd.read_parquet(f"{BENCH}/lecturas.parquet"))

print(f"read_csv     : {t_csv} s")
print(f"read_parquet : {t_pq} s")

Parquet gana, pero **no es para tirar cohetes**. Si la demo acabara aquí, el argumento sería
flojo: *"un poco más rápido y ocupa menos"*.

El argumento de verdad es otro. **Vuelve a la pregunta de Movilidad**: *NO₂ medio por estación*.
Para responderla necesito **2 columnas de las 6**. Las otras cuatro me sobran.

In [ ]:
t_col, d_col = cronometra(
    lambda: pd.read_parquet(f"{BENCH}/lecturas.parquet", columns=["estacion", "no2"])
)
print(f"read_parquet (2 columnas de 6): {t_col} s")
print()
print(f"   vs Parquet entero : {round(t_pq/t_col, 1)}x más rápido")
print(f"   vs CSV            : {round(t_csv/t_col, 1)}x más rápido")

Fíjate en lo que **no** ha pasado: no he leído la tabla y filtrado columnas después.
**No las he leído.** No han tocado el disco.

El CSV no puede hacer esto ni queriendo: guarda **fila a fila**, así que para llegar a la
columna 6 de la fila un millón tiene que atravesar las cinco anteriores, un millón de veces.
Parquet guarda **columna a columna**: si le pides dos, abre dos bloques y se va.

Eso es *almacenamiento columnar*. No es una etiqueta de marketing: es la razón física de ese
número que acabas de ver.

> **Y aquí está el apartado del examen que casi nadie contesta bien: el económico.** En la nube
> no pagas solo por lo que guardas. Pagas por **lo que escaneas**. Leer 2 columnas de 6 no es
> solo más rápido: es una factura menor, cada día, para siempre. Un formato no es una
> preferencia estética. Es una línea en el presupuesto.

### El matiz honesto (el que separa un 6 de un 9)

In [ ]:
print("Parquet en disco :", mb(f"{BENCH}/lecturas.parquet"), "MB")
print("El mismo df en RAM:", round(d_pq.memory_usage(deep=True).sum() / 1024**2, 1), "MB")

Parquet arrasa en disco. Pero en cuanto lo abres en pandas **se descomprime en memoria**: el
ahorro es de **disco y de red**, no de RAM.

Decir *"Parquet ocupa menos"* sin este matiz es media respuesta.

**Ya tenemos el formato.** Y ya podríamos responder a Movilidad... hasta mañana. Porque mañana
alguien va a ejecutar un pipeline con un bug encima de esta tabla.

---
# ACTO 3 · El día que alguien machaca la tabla buena

Esto no es hipotético: es martes por la tarde. Un compañero lanza un proceso mal filtrado y
**sobrescribe** el histórico. Con Parquet a secas, esos datos ya no existen. Y con ellos, tu
respuesta.

Vamos a guardar lo mismo, pero como **tabla Delta**. Presta atención a lo que aparece en el
disco: es lo único que cambia, y lo cambia todo.

In [ ]:
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake

D = str(PROC / "lecturas_delta")
shutil.rmtree(D, ignore_errors=True)

df_hist = pd.read_parquet(f"{BENCH}/lecturas.parquet")
write_deltalake(D, pa.Table.from_pandas(df_hist, preserve_index=False), mode="overwrite")

print("Escrita la versión 0 con", f"{len(df_hist):,}", "filas.")

In [ ]:
for root, _, files in os.walk(D):
    for f in sorted(files):
        print(os.path.join(root, f).replace(D, "lecturas_delta"))

**Míralo bien: son ficheros Parquet normales y corrientes.** Los mismos de hace diez minutos.

Lo único que Delta ha añadido es esa carpeta: **`_delta_log/`**. Y dentro no hay magia.
Hay esto:

In [ ]:
with open(f"{D}/_delta_log/00000000000000000000.json") as f:
    for linea in f:
        d = json.loads(linea)
        clave = list(d)[0]
        print(f"--- {clave} ---")
        print(json.dumps(d[clave], indent=2)[:400], "\n")

Tres líneas de JSON que dicen: *"la versión 0 de esta tabla **son estos ficheros**, con **este
esquema**, escritos a **esta hora**"*.

Eso es todo. **Eso** es lo que convierte una carpeta de ficheros sueltos en una **tabla**.

Ahora viene el martes por la tarde.

In [ ]:
# El compañero lanza su proceso mal filtrado y machaca la tabla.
df_roto = df_hist[df_hist["no2"].between(0, 50)]     # se deja fuera media ciudad
write_deltalake(D, pa.Table.from_pandas(df_roto, preserve_index=False), mode="overwrite")

dt = DeltaTable(D)
print("versión actual :", dt.version())
print("filas ahora    :", f"{len(df_roto):,}", " <-- nos hemos dejado por el camino", f"{len(df_hist)-len(df_roto):,}")

In [ ]:
# ¿Qué ha pasado en el disco?
for root, _, files in os.walk(D):
    for f in sorted(files):
        print(os.path.join(root, f).replace(D, "lecturas_delta"))

**Dos cosas que merecen tu atención:**

1. Hay **dos** ficheros de log: `...0000.json` y `...0001.json`. El log **no se reescribe:
   añade**.
2. Y hay **dos** ficheros Parquet. **El viejo sigue ahí.** El "overwrite" no ha borrado nada.

¿Qué dice el log nuevo?

In [ ]:
with open(f"{D}/_delta_log/00000000000000000001.json") as f:
    for linea in f:
        d = json.loads(linea)
        clave = list(d)[0]
        if clave in ("add", "remove"):
            print(f"{clave.upper():>7} -> {d[clave]['path']}")

Ahí está el mecanismo entero, en dos palabras: **`remove` y `add`**.

El "borrado" no borró: **escribió una línea que dice "este fichero ya no cuenta"**. El fichero
sigue en el disco. Y si sigue en el disco...

In [ ]:
antigua = DeltaTable(D, version=0).to_pandas()
# (Si tu versión de deltalake se queja, la alternativa es:
#   dt = DeltaTable(D); dt.load_as_version(0); antigua = dt.to_pandas())

print("filas en la versión actual (1):", f"{len(df_roto):,}")
print("filas en la versión 0        :", f"{len(antigua):,}", " <-- están todas")

In [ ]:
dt.history()   # el diario completo de la tabla

**Acabo de recuperar un millón de filas que había "borrado" hace treinta segundos.**

Eso es *time travel*, y no es un truco de feria: es la **consecuencia directa** de que el log
añade en vez de reescribir. Del mismo mecanismo salen las otras garantías:

- **Atomicidad**: si el proceso se cae a mitad, la línea del log no se escribe → nadie lee
  media tabla. O está entera, o no está.
- **Evolución de esquema**: el esquema vive en el log, no en cada fichero.
- **Auditoría**: `history()` te dice quién escribió qué y cuándo.

Esas cuatro letras que verás en la teoría, **ACID**, son esto. No un adjetivo: un fichero JSON.

> En tu actividad hay una línea que te pide `delta_version`. Ya sabes qué número es ése y de
> dónde sale.

**Ahora sí: la tabla es fiable.** Podemos responder a Movilidad... y justo entonces llama el
equipo de la app del ciudadano.

---
# ACTO 4 · *"¿Cómo está MI calle AHORA?"*

La app móvil necesita responder eso en **20 milisegundos**, para **un** ciudadano, con **todos**
los datos de su estación.

Y nuestra preciosa tabla Delta **es malísima para eso**. No porque esté mal hecha: porque es
**otro problema**.

> **Requiere Docker + pymongo.** Es el bloque opcional de la demo:
> ```bash
> pip install -r demo/nf1_smartcity/requirements-demo.txt
> docker run -d --name mongo-demo -p 27017:27017 mongo:7
> ```

In [ ]:
api = json.load(open(f"{RAW}/meteorologia.json", encoding="utf-8"))
print(json.dumps(api[0], indent=2, ensure_ascii=False))   # JSON anidado, tal cual lo da la API

In [ ]:
from pymongo import MongoClient

col = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=3000).ciudad.tiempo
col.delete_many({})
col.insert_many(api)
print("insertados:", col.count_documents({}))

**Gesto 1 — Encaje natural.** No he aplanado nada. No he creado tablas. No he definido un
esquema. **He guardado el JSON tal cual.**

Compáralo con lo que harás en la Fase 1 de tu actividad con `json_normalize`: eso está bien
**porque el destino es analítico**. Aquí el destino es otro.

In [ ]:
col.create_index("id_estacion")

t = time.perf_counter()
doc = col.find_one({"id_estacion": 17})
ms = (time.perf_counter() - t) * 1000

print(f"documento completo en {ms:.1f} ms")
print(json.dumps(doc, indent=2, ensure_ascii=False, default=str))

**Gesto 2 — Latencia.** Milisegundos, y me ha devuelto el **documento entero**.

Tu data lake haría esto fatal: tendría que abrir *footers* y rebuscar en bloques. Es rapidísimo
escaneando millones de filas y torpe buscando una. **No es peor: es otra cosa.**

In [ ]:
# Gesto 3: la API ha añadido un campo esta mañana. Lo insertamos sin avisar a nadie.
col.insert_one({"id_estacion": 999, "nombre": "EST-NUEVA",
                "medida": {"temp": 21.4}, "calidad_aire_ICA": "buena"})   # <- campo nuevo

print(col.find_one({"id_estacion": 999}))

**Gesto 3 — Flexibilidad de esquema.** No ha pasado **nada**. Ni un error, ni una migración,
ni un aviso.

En una relacional eso es un `ALTER TABLE` a las tres de la mañana. En un data lake, un fichero
que ya no cuadra con los demás. Esto es la respuesta literal a una pregunta que ha caído en
convocatoria: *¿dónde guardarías las respuestas JSON de una API y por qué?*

> **Lo importante, y lo que casi nadie escribe en el examen:** la arquitectura real **es la
> suma, no la elección**. NoSQL para lo *caliente* (lo que la app necesita ahora mismo),
> lakehouse para lo *frío* (el histórico que responde a Movilidad), y una copia del JSON crudo
> cae igualmente en `raw/` para no perder nada. Cuando te pregunten *"¿SQL o NoSQL?"*, la
> respuesta de sobresaliente **no elige bando**: explica qué va en cada sitio y por qué.
>
> MongoDB **no** entra en tu actividad ni en la entrega. No lo instales.

In [ ]:
# Al terminar la demo:  docker stop mongo-demo && docker rm mongo-demo
print("fin del bloque opcional")

---
# ACTO 5 · La respuesta

Volvemos al minuto 1. *"¿Tenemos un problema de NO₂ en el Ensanche?"*

Ahora sí.

In [ ]:
hist = pd.read_parquet(f"{BENCH}/lecturas.parquet", columns=["estacion", "no2"])
hist.groupby("estacion")["no2"].mean().round(1).sort_values(ascending=False)

**Sí. El Ensanche está claramente peor.** Y esto ya no es una tabla con `Ensanche` y `ensanche`
peleándose: son cuatro estaciones, un número cada una.

Pero esto todavía es una **observación**. Y una observación no cambia nada. Vamos a mirar
*cuándo*.

In [ ]:
h = pd.read_parquet(f"{BENCH}/lecturas.parquet", columns=["timestamp", "estacion", "no2"])
h["hora"] = pd.to_datetime(h["timestamp"]).dt.hour

por_hora = h[h["estacion"] == "ensanche"].groupby("hora")["no2"].mean().round(1)
por_hora.plot(kind="bar", figsize=(11, 3), title="Ensanche · NO₂ medio por hora del día");

### Observación → *insight* → decisión

- **Observación** *(no sirve para nada por sí sola)*: "el Ensanche tiene el NO₂ más alto".
- ***Insight*** *(esto ya es trabajo de ingeniero)*: hay **dos fenómenos distintos** mezclados.
  Un **suelo permanente** de ~63 µg/m³ que no baja **ni a las cuatro de la mañana** — el
  Ensanche está el doble de contaminado que el Norte **las 24 horas** —, y encima **dos picos**
  de tráfico (8 h y 19 h) que suman ~20 más durante unas pocas horas.
- **Decisión** *(y aquí está la honestidad profesional)*: restringir el tráfico en hora punta
  ataca **los picos**, que son la parte pequeña. **El problema de fondo del Ensanche es
  estructural y no se arregla con un horario.** Si me hubiera quedado en la media por estación,
  habría recomendado la medida equivocada con toda la seguridad del mundo.

> Eso es lo que separa a alguien que sabe pandas de alguien que hace ingeniería de datos. Y es,
> literalmente, un apartado del examen: **distinguir una observación de un *insight*
> accionable**.

---

## Lo que ha pasado en esta hora

Una pregunta. Cuatro obstáculos. Cada uno te obligaba al siguiente:

| | El obstáculo | La herramienta | Dónde está en la teoría |
|---|---|---|---|
| 1 | El dato crudo miente | Diagnóstico y reglas de limpieza | **§1.6** (📖 de consulta) |
| 2 | A escala, el formato es dinero | Parquet / columnar | **§1.4** + vídeo 2.3 |
| 3 | Alguien machaca la tabla | Delta y su log de transacciones | **§1.5** + `VIDEO_NF1_B` |
| 4 | La app quiere un dato, no un millón | NoSQL documental | **§1.7** (📖) |
| 5 | Un número no es una decisión | Observación → *insight* | **§1.8** |

**Y ahora te toca a ti**, con los datos de un juego: sesiones, partidas y matchmaking. Mismo
trabajo, otros datos. Empieza por `VIDEO_NF1_A`.